In [2]:
"""
LUNA16 masked-volume extraction for false-positive-reduction (Dou et al. 2017).

Strategy (per subset folder):
  - read each .mhd/.raw scan
  - resample to 1x1x1 mm isotropic
  - build a "keep" mask = union of all candidate boxes (positives AND all
    negatives) of size KEEP_BOX voxels, centred on each candidate
  - zero out every voxel OUTSIDE that union, then save the whole resampled
    volume with np.savez_compressed -- the zeroed sea compresses to almost
    nothing, the candidate neighbourhoods are kept once, at full fidelity
  - record per-scan geometry + every candidate coordinate in one master CSV

At training time: load a volume, look up its candidates in the CSV, and cut
the 48^3 (or multi-scale) cubes on the fly. Every voxel stored exactly once,
no per-candidate duplication, all negatives kept for free.

After this runs delete the source subset folder.
"""

import os
import glob
import numpy as np
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm
from config import DATA_DIR
from pathlib import Path

# ----------------------------------------------------------------------
# CONFIG -- adjust these
# ----------------------------------------------------------------------
SUBSET_DIR      = Path(DATA_DIR) / "subset1"                 # folder with the .mhd/.raw files
CANDIDATES_CSV  = Path(DATA_DIR) / "candidates_V2.csv"       # LUNA16 candidate list
OUT_DIR         = Path(DATA_DIR) / "masked_scans1"            # where volumes + master CSV are written

NEW_SPACING     = np.array([1.0, 1.0, 1.0]) # target isotropic spacing (x,y,z) mm

# KEEP_BOX is the half-size (voxels) of the box kept around each candidate.
# Must be >= half of the largest cube you will EVER cut at training time.
# Archi-3 needs 40x40x26; with margin for augmentation a 48^3 training cube is
# typical, so we keep +/-28 (=> 56 voxels) around every candidate to be safe.
KEEP_HALF       = 28
OUTSIDE_VALUE   = -1000                     # HU of air; what the masked-out sea is set to
DTYPE           = np.int16                  # HU are integers; int16 halves disk vs float32
# ----------------------------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)


def load_itk(path):
    """Load an .mhd scan. Returns (array[z,y,x], origin[x,y,z], spacing[x,y,z])."""
    img     = sitk.ReadImage(path)
    array   = sitk.GetArrayFromImage(img)            # (z, y, x)
    origin  = np.array(img.GetOrigin())              # (x, y, z) mm
    spacing = np.array(img.GetSpacing())             # (x, y, z) mm
    return array, origin, spacing


def resample(array, old_spacing, new_spacing):
    """Resample [z,y,x] volume to new isotropic spacing. spacings are [x,y,z].
    Returns (resampled_array, real_factor[z,y,x])."""
    from scipy.ndimage import zoom
    old_zyx = old_spacing[::-1]
    new_zyx = new_spacing[::-1]
    resize_factor = old_zyx / new_zyx
    new_shape = np.round(np.array(array.shape) * resize_factor)
    real_factor = new_shape / np.array(array.shape)
    resampled = zoom(array, real_factor, order=1)    # linear interpolation
    return resampled, real_factor


def world_to_voxel(world_xyz, origin, spacing):
    """World (mm) -> voxel index (x, y, z) in the resampled volume."""
    return np.round((np.array(world_xyz) - origin) / spacing).astype(int)


def main():
    cands = pd.read_csv(CANDIDATES_CSV)

    mhd_files = sorted(glob.glob(os.path.join(SUBSET_DIR, "*.mhd")))
    print(f"Found {len(mhd_files)} scans in {SUBSET_DIR}")

    records = []          # one row per candidate, for the master CSV
    kept_total = 0
    full_total = 0

    pbar = tqdm(mhd_files, desc="Processing scans", unit="scan")
    for mhd_path in pbar:
        seriesuid = os.path.splitext(os.path.basename(mhd_path))[0]
        array, origin, old_spacing = load_itk(mhd_path)

        # 1) resample to 1mm isotropic (origin is unchanged by resampling)
        array_iso, _ = resample(array, old_spacing, NEW_SPACING)
        array_iso = array_iso.astype(DTYPE)
        nz, ny, nx = array_iso.shape

        # 2) build the keep-mask: union of candidate boxes
        scan_cands = cands[cands.seriesuid == seriesuid]
        keep = np.zeros(array_iso.shape, dtype=bool)   # [z,y,x]

        for _, row in scan_cands.iterrows():
            vx, vy, vz = world_to_voxel(
                (row.coordX, row.coordY, row.coordZ), origin, NEW_SPACING)
            z0, z1 = max(vz - KEEP_HALF, 0), min(vz + KEEP_HALF, nz)
            y0, y1 = max(vy - KEEP_HALF, 0), min(vy + KEEP_HALF, ny)
            x0, x1 = max(vx - KEEP_HALF, 0), min(vx + KEEP_HALF, nx)
            keep[z0:z1, y0:y1, x0:x1] = True

            records.append({
                "seriesuid": seriesuid,
                "class":     int(row["class"]),
                # voxel index in THIS saved volume (origin/spacing below let you
                # re-derive it, but storing it directly saves a step at train time)
                "voxel_x":   int(vx),
                "voxel_y":   int(vy),
                "voxel_z":   int(vz),
                "coordX":    row.coordX,
                "coordY":    row.coordY,
                "coordZ":    row.coordZ,
            })

        # 3) zero everything outside the union, then save compressed
        array_iso[~keep] = OUTSIDE_VALUE

        out_path = os.path.join(OUT_DIR, f"{seriesuid}.npz")
        np.savez_compressed(
            out_path,
            volume=array_iso,                 # [z,y,x], int16, masked
            origin=origin.astype(np.float64), # (x,y,z) mm of voxel (0,0,0)
            spacing=NEW_SPACING.astype(np.float64),  # (x,y,z) mm, = 1,1,1
            shape=np.array(array_iso.shape),  # (z,y,x)
        )

        # bookkeeping for the kept-fraction report
        kept = int(keep.sum())
        full = array_iso.size
        kept_total += kept
        full_total += full
        n_pos = int((scan_cands["class"] == 1).sum())
        n_neg = int((scan_cands["class"] == 0).sum())
        pbar.set_postfix_str(
            f"{seriesuid[-8:]}  {n_pos}p/{n_neg}n  kept={100*kept/full:4.1f}%")

    # master coordinate CSV (the "header file")
    meta = pd.DataFrame(records)
    meta_path = os.path.join(OUT_DIR, "candidates_index.csv")
    meta.to_csv(meta_path, index=False)

    print(f"\nDone. {len(mhd_files)} masked volumes written to {OUT_DIR}/")
    print(f"Master index: {meta_path}")
    print(f"  candidates: {len(meta)}  "
          f"(pos {int((meta['class']==1).sum())}, neg {int((meta['class']==0).sum())})")
    print(f"  mean kept fraction of volume: {100*kept_total/full_total:.1f}%")
    print("  (compression shrinks the masked-out part further on disk)")


if __name__ == "__main__":
    main()

Found 89 scans in C:\Users\felix\Studium\Ausland\ERASMUS\Data_analysis_ML\LUNA_16\subset1


Processing scans: 100%|██████████| 89/89 [07:00<00:00,  4.72s/scan, 54369557  1p/965n  kept=46.9%]  



Done. 89 masked volumes written to C:\Users\felix\Studium\Ausland\ERASMUS\Data_analysis_ML\LUNA_16\masked_scans1/
Master index: C:\Users\felix\Studium\Ausland\ERASMUS\Data_analysis_ML\LUNA_16\masked_scans1\candidates_index.csv
  candidates: 71012  (pos 170, neg 70842)
  mean kept fraction of volume: 31.1%
  (compression shrinks the masked-out part further on disk)
